In [ ]:
# Qdrant
from qdrant_client import QdrantClient
print(QdrantClient(url="http://localhost:6333").get_collections())

In [ ]:
"""
### Ollama / Llama 3.1
from langchain_ollama import ChatOllama
print(ChatOllama(model="llama3.1").invoke("ok?"))
"""

"""
### Ollama / Llama 3.1 (con streaming)
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1")

# Stampa i token man mano che vengono generati
for chunk in llm.stream("ok?"):
    print(chunk.content, end="", flush=True)
print()
"""

### Ollama / Llama 3.2: (con streaming)
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    keep_alive="1h",
    num_thread=4 #fondamentale per evitare bottleneck
)

for chunk in llm.stream("Rispondi con una frase: cos'è un grafo?"):
    print(chunk.content, end="", flush=True)

In [ ]:
# NetworkX
import networkx as nx
G = nx.Graph(); G.add_edge("a", "b", weight=0.9)
print(nx.node_link_data(G))

In [1]:
# anywidget
import anywidget, traitlets
from IPython.display import display

class _T(anywidget.AnyWidget):
    _esm = "function render({el}){ el.innerHTML = '✅ anywidget ok'; } export default {render};"
_T()
display(_T())

In [3]:
# anywidget > grafo di prova
import anywidget
import traitlets
import networkx as nx

# 1. Creazione del grafo di test NetworkX
G = nx.Graph()
G.add_node("chunk_a", text="GraphRAG e Vector DB")
G.add_node("chunk_b", text="Qdrant per la ricerca")
G.add_node("chunk_c", text="NetworkX per la struttura")
G.add_edge("chunk_a", "chunk_b", weight=0.9)
G.add_edge("chunk_a", "chunk_c", weight=0.75)

# Convertiamo il grafo NetworkX in JSON compatibile
graph_json = nx.node_link_data(G)
# Se nx.node_link_data genera la chiave 'edges', la rinominiamo in 'links' per D3.js
if "edges" in graph_json:
    graph_json["links"] = graph_json.pop("edges")

# 2. Definizione della Classe AnyWidget con D3.js integrato
class GraphWidget(anywidget.AnyWidget):
    _esm = """
    import * as d3 from "https://esm.sh/d3@7";

    export function render({ model, el }) {
      el.innerHTML = "";
      
      const width = 600;
      const height = 350;
      
      const svg = d3.select(el).append("svg")
        .attr("width", width)
        .attr("height", height)
        .style("background", "#f8fafc")
        .style("border", "1px solid #cbd5e1")
        .style("border-radius", "8px");

      function draw() {
        const graph = model.get("graph_data");
        if (!graph || !graph.nodes || graph.nodes.length === 0) return;

        svg.selectAll("*").remove();

        const nodes = graph.nodes.map(d => ({ ...d }));
        const links = graph.links ? graph.links.map(d => ({ ...d })) : [];

        const simulation = d3.forceSimulation(nodes)
          .force("link", d3.forceLink(links).id(d => d.id).distance(100))
          .force("charge", d3.forceManyBody().strength(-180))
          .force("center", d3.forceCenter(width / 2, height / 2));

        const link = svg.append("g")
          .selectAll("line")
          .data(links)
          .enter().append("line")
          .attr("stroke", "#94a3b8")
          .attr("stroke-width", 2);

        const node = svg.append("g")
          .selectAll("circle")
          .data(nodes)
          .enter().append("circle")
          .attr("r", 12)
          .attr("fill", "#6366f1")
          .style("cursor", "pointer")
          .on("click", (event, d) => {
             model.set("selected_node", { id: d.id, text: d.text || "" });
             model.save_changes();
          });

        const label = svg.append("g")
          .selectAll("text")
          .data(nodes)
          .enter().append("text")
          .text(d => d.id)
          .attr("font-size", "11px")
          .attr("dx", 15)
          .attr("dy", 4);

        simulation.on("tick", () => {
          link
            .attr("x1", d => d.source.x)
            .attr("y1", d => d.source.y)
            .attr("x2", d => d.target.x)
            .attr("y2", d => d.target.y);

          node
            .attr("cx", d => d.x)
            .attr("cy", d => d.y);

          label
            .attr("x", d => d.x)
            .attr("y", d => d.y);
        });
      }

      model.on("change:graph_data", draw);
      draw();
    }
    """
    
    # Proprietà reattive sincronizzate
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_node = traitlets.Dict({}).tag(sync=True)

# 3. Istanziazione del widget e caricamento dati
widget = GraphWidget()
widget.graph_data = graph_json
widget

In [4]:
# anywidget -- classi collegate
import sys
from pathlib import Path

# Aggiunge la cartella radice del progetto al sys.path per importare da src/
sys.path.append(str(Path.cwd()))

from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

# 1. Costruzione del grafo tramite la classe modulare
builder = KnowledgeGraphBuilder()
builder.add_chunk_node("chunk_0", text="GraphRAG combina Vector DB e Grafi di Conoscenza.")
builder.add_chunk_node("chunk_1", text="Qdrant gestisce la ricerca vettoriale ad alte prestazioni.")
builder.add_chunk_node("chunk_2", text="NetworkX modella la struttura del grafo nel backend Python.")

builder.add_relation("chunk_0", "chunk_1", weight=0.85)
builder.add_relation("chunk_0", "chunk_2", weight=0.78)

# 2. Istanziazione e visualizzazione del widget
widget = ChunkGraphWidget()
widget.graph_data = builder.to_json_data()
widget